In [ ]:
import pandas as pd
import numpy as np

import torch
from torch_geometric.data import Data 

from torch_geometric.loader import DataLoader
import torch.nn as nn
from torch_geometric.nn import GATConv
from torch_geometric.nn import global_mean_pool, global_max_pool
import torch.optim as optim
from torch.utils.data import random_split

import os 

In [2]:
df = pd.read_csv(r"C:\Projects\SPD-Track-Parameter-Prediction\data\processed\processed_signal_hits_scaled.tsv", sep = "\t")
df.head()

,event_id,x,y,z,station,track_id,px,py,pz,vertex_x,vertex_y,vertex_z,pt,phi,theta,charge,radius,phi_hit,rho_z,momentum
0,0,-0.600199,-0.286108,0.264916,-1.667500,0,-1.214894,-0.667237,1.121487,-23.817434,-17.20341,-176.948134,0.176548,3.533347,2.485765,-1,-1.667981,-1.495022,-1.217578,0.416363
1,0,-0.636314,-0.306125,0.299236,-1.568377,0,-1.210494,-0.675384,1.121487,-23.817434,-17.20341,-176.948134,0.176548,3.533347,2.485765,-1,-1.569035,-1.493115,-1.147679,0.416363
2,0,-0.672756,-0.326168,0.333678,-1.469255,0,-1.206040,-0.683503,1.121487,-23.817434,-17.20341,-176.948134,0.176548,3.533347,2.485765,-1,-1.469333,-1.491514,-1.076815,0.416363
3,0,-0.708468,-0.346751,0.368289,-1.370132,0,-1.201531,-0.691593,1.121487,-23.817434,-17.20341,-176.948134,0.176548,3.533347,2.485765,-1,-1.370632,-1.489510,-1.005695,0.416363
4,0,-0.744825,-0.367197,0.402689,-1.271010,0,-1.196969,-0.699654,1.121487,-23.817434,-17.20341,-176.948134,0.176548,3.533347,2.485765,-1,-1.270662,-1.487984,-0.933955,0.416363


In [3]:
event_0 = df[df["event_id"] == 0]
event_0

,event_id,x,y,z,station,track_id,px,py,pz,vertex_x,vertex_y,vertex_z,pt,phi,theta,charge,radius,phi_hit,rho_z,momentum
0,0,-0.600199,-0.286108,0.264916,-1.667500,0,-1.214894,-0.667237,1.121487,-23.817434,-17.20341,-176.948134,0.176548,3.533347,2.485765,-1,-1.667981,-1.495022,-1.217578,0.416363
1,0,-0.636314,-0.306125,0.299236,-1.568377,0,-1.210494,-0.675384,1.121487,-23.817434,-17.20341,-176.948134,0.176548,3.533347,2.485765,-1,-1.569035,-1.493115,-1.147679,0.416363
2,0,-0.672756,-0.326168,0.333678,-1.469255,0,-1.206040,-0.683503,1.121487,-23.817434,-17.20341,-176.948134,0.176548,3.533347,2.485765,-1,-1.469333,-1.491514,-1.076815,0.416363
3,0,-0.708468,-0.346751,0.368289,-1.370132,0,-1.201531,-0.691593,1.121487,-23.817434,-17.20341,-176.948134,0.176548,3.533347,2.485765,-1,-1.370632,-1.489510,-1.005695,0.416363
4,0,-0.744825,-0.367197,0.402689,-1.271010,0,-1.196969,-0.699654,1.121487,-23.817434,-17.20341,-176.948134,0.176548,3.533347,2.485765,-1,-1.270662,-1.487984,-0.933955,0.416363
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205,0,1.837082,0.389286,-0.954152,1.306176,5,0.857590,-0.035604,-0.293048,-23.817434,-17.20341,-176.948134,-0.679735,0.453250,2.071306,1,1.306038,0.112426,0.555713,-0.645530
206,0,1.879693,0.387397,-0.969339,1.405299,5,0.857180,-0.045160,-0.293048,-23.817434,-17.20341,-176.948134,-0.679735,0.453250,2.071306,1,1.405440,0.109382,0.607349,-0.645530
207,0,1.922367,0.385032,-0.984033,1.504421,5,0.856665,-0.054725,-0.293048,-23.817434,-17.20341,-176.948134,-0.679735,0.453250,2.071306,1,1.504902,0.106332,0.658516,-0.645530
208,0,1.964353,0.382336,-0.999285,1.603544,5,0.856046,-0.064297,-0.293048,-23.817434,-17.20341,-176.948134,-0.679735,0.453250,2.071306,1,1.602727,0.103354,0.709690,-0.645530


In [5]:
def build_trk_grph(trk_df):
    
    trk_df = trk_df.sort_values(by="station")
    trk_df = trk_df.reset_index(drop=True)

    feature_columns=["x","y","z","radius","phi_hit","rho_z","station"]


    node_features = trk_df[feature_columns].values


    x = torch.tensor(node_features,dtype=torch.float)

    edges=[]

    for i in range(len(trk_df)):
        for j in range(i+1,min(i+3,len(trk_df))):

            edges.append([i,j])
            edges.append([j,i])


    edge_index = torch.tensor(edges,dtype=torch.long).t().contiguous()

    label_columns=["px","py","pz",#"pt","momentum",#"phi",#"theta",#"vertex_z"
]

    #label = trk_df[label_columns].mean().values
    label = trk_df.iloc[0][label_columns].values

    y = torch.tensor(label,dtype=torch.float).unsqueeze(0)

    graph = Data(x=x,edge_index=edge_index,y=y)

    return graph

In [ ]:
def build_graph_dataset(df):
    graphs = []
    event_ids = df["event_id"].unique()
    for event_id in event_ids:
        event_df = df[df["event_id"]==event_id]
        trk_ids = event_df["track_id"].unique()
        for track_id in trk_ids:
            trk_df = event_df[event_df["track_id"]==track_id]
            graph = build_trk_grph(trk_df)
            graphs.append(graph)
    

    return graphs

In [7]:
graphs = build_graph_dataset(df)
print(len(graphs))


24984


In [8]:
print(graphs[0])
print(graphs[0].x)
print(graphs[0].y)


Data(x=[35, 7], edge_index=[2, 134], y=[1, 3])
tensor([[-0.6002, -0.2861,  0.2649, -1.6680, -1.4950, -1.2176, -1.6675],
        [-0.6363, -0.3061,  0.2992, -1.5690, -1.4931, -1.1477, -1.5684],
        [-0.6728, -0.3262,  0.3337, -1.4693, -1.4915, -1.0768, -1.4693],
        [-0.7085, -0.3468,  0.3683, -1.3706, -1.4895, -1.0057, -1.3701],
        [-0.7448, -0.3672,  0.4027, -1.2707, -1.4880, -0.9340, -1.2710],
        [-0.7802, -0.3883,  0.4376, -1.1721, -1.4859, -0.8619, -1.1719],
        [-0.8160, -0.4094,  0.4720, -1.0726, -1.4842, -0.7897, -1.0728],
        [-0.8512, -0.4309,  0.5068, -0.9738, -1.4823, -0.7172, -0.9736],
        [-0.8866, -0.4526,  0.5413, -0.8746, -1.4804, -0.6447, -0.8745],
        [-0.9215, -0.4747,  0.5760, -0.7760, -1.4784, -0.5720, -0.7754],
        [-0.9569, -0.4966,  0.6105, -0.6762, -1.4768, -0.4989, -0.6763],
        [-0.9918, -0.5191,  0.6451, -0.5772, -1.4749, -0.4260, -0.5772],
        [-1.0265, -0.5417,  0.6798, -0.4780, -1.4731, -0.3527, -0.4780],
    

In [14]:
invalid_grphs = 0
for graph in graphs:
    if graph.x.shape[1] !=7:
        invalid_grphs +=1
print("Invalid Graphs :", invalid_grphs)

Invalid Graphs : 0


In [47]:
invalid_labels = 0

for graph in graphs:

    if graph.y.shape[1] != 3:

        invalid_labels += 1
print("Invalid Labels : ", invalid_labels)

Invalid Labels :  0


In [11]:

torch.manual_seed(42)
np.random.seed(42)
# data split
dataset_size = len(graphs)

train_size = int(0.70 * dataset_size)

val_size = int(0.15 * dataset_size)

test_size = (dataset_size- train_size- val_size)

train_dataset, val_dataset, test_dataset = random_split(graphs,[train_size,val_size,test_size])

In [15]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

17488
3747
3749


In [16]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)

In [17]:
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = False)

In [18]:
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False)

In [19]:
batch = next(iter(train_loader))
print(batch)

DataBatch(x=[1108, 7], edge_index=[2, 4240], y=[32, 3], batch=[1108], ptr=[33])


In [20]:
print(batch.batch)
print(batch.batch[:50])
print(batch.num_graphs)
print(batch.num_nodes)
print(batch.num_edges)

tensor([ 0,  0,  0,  ..., 31, 31, 31])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1])
32
1108
4240


In [38]:
class trackGAT(nn.Module):

    def __init__(self):
        super().__init__()
        self.gat1 = GATConv(7, 128,heads=1,concat=False )
        self.gat2 = GATConv(128, 128, heads=1, concat=False)
        self.dropout = nn.Dropout(0.2)
        self.linear = nn.Sequential(
            nn.Linear(256,64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,3)
        )


    def forward(self, x,edge_index, batch):
        x = self.gat1(x,edge_index)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.gat2(x, edge_index)

        x = torch.relu(x)

        x1 = global_mean_pool(x,batch )
        x2 = global_max_pool( x,batch)

        x = torch.cat([x1,x2],dim=1)

        x = self.linear(x)
        return x

In [39]:
model = trackGAT()
batch = next(iter(train_loader))
prediction = model(batch.x, batch.edge_index, batch.batch)
print(prediction.shape)
# print(batch.batch)


torch.Size([32, 3])


In [40]:

# loss function 

print(graphs[0].y)
print(graphs[0].y.shape)

batch = next(iter(train_loader))
prediction = model(batch.x, batch.edge_index, batch.batch)

criterion = nn.MSELoss()

loss = criterion(prediction, batch.y)

print(loss)

print(prediction.shape)
print(batch.y.shape)

tensor([[-1.2149, -0.6672,  1.1215]])
torch.Size([1, 3])
tensor(1.4299, grad_fn=<MseLossBackward0>)
torch.Size([32, 3])
torch.Size([32, 3])


In [41]:
batch = next(iter(train_loader))

print(batch)

print(batch.y.shape)

print(batch.y)

print(batch.num_graphs)

DataBatch(x=[1085, 7], edge_index=[2, 4148], y=[32, 3], batch=[1085], ptr=[33])
torch.Size([32, 3])
tensor([[-3.1945e-01,  5.0393e-01, -2.4353e+00],
        [ 6.1663e-01, -2.5148e-01,  2.6674e-01],
        [-5.2632e-01,  1.0575e+00, -2.9271e-03],
        [-6.8241e-01,  8.1940e-01, -2.8710e-01],
        [-2.9849e-01,  2.0793e+00,  2.7458e-01],
        [-6.4757e-01,  1.5695e+00,  8.2008e-01],
        [ 2.1776e-02,  7.7665e-01, -5.0865e-01],
        [ 4.3457e-02,  6.0227e-01, -2.0969e-01],
        [-2.8144e-01, -1.7023e-01, -3.4805e-02],
        [-8.5736e-02, -2.0747e+00, -5.3857e-01],
        [-2.0570e+00, -7.4605e-01,  4.8160e-01],
        [-1.8032e-01, -1.4001e+00, -1.2432e+00],
        [ 6.8218e-02, -1.8359e+00, -2.3521e-01],
        [ 1.8733e+00, -1.2136e+00,  1.1643e+00],
        [-1.2112e+00,  1.6501e+00,  3.5981e+00],
        [ 1.9299e+00,  4.3479e-01, -5.6254e-01],
        [-1.2821e+00, -3.3058e-02, -1.7113e+00],
        [ 3.4343e-01,  6.6526e-01,  1.5313e-01],
        [-2.5981e-

In [42]:
# optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.0003,
    weight_decay=1e-5
)
print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0003
    maximize: False
    weight_decay: 1e-05
)


In [43]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience=5,
    factor=0.5
)

In [ ]:
num_epochs = 80

model_save_dir = r"C:\Projects\SPD-Track-Parameter-Prediction\models"
os.makedirs(model_save_dir, exist_ok = True)
best_model_path = os.path.join(model_save_dir, "px_py_pz_best_model.pt")
val_loss_min = float('inf')

# training loop
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        
        
        optimizer.zero_grad()
        prediction = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(prediction, batch.y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() # .item() extracts the python number from the tensor
    avg_train_loss = train_loss / len(train_loader)
    
    # val loop
    
    model.eval()

    total_validation_loss = 0

    with torch.no_grad():

        for batch in val_loader:

            prediction = model(
                batch.x,
                batch.edge_index,
                batch.batch
            )
            
            loss = criterion(
                prediction,
                batch.y
            )

            total_validation_loss += loss.item()

    avg_validation_loss = float(total_validation_loss) / len(val_loader)
    
    
    
    if avg_validation_loss < val_loss_min:
        
        print(f" Validation loss decreased ({val_loss_min:.4f} to {avg_validation_loss:.4f}) Saving best model")
        val_loss_min = avg_validation_loss
        torch.save({'epoch': epoch,
                   'model_state_dictionary': model.state_dict(),
                   'optimizer_state_dictionary': optimizer.state_dict(),'val_loss_min': val_loss_min}, best_model_path)
    scheduler.step(avg_validation_loss)
    
    print(
        f"Epoch {epoch+1}/{num_epochs}"
        f"      Train Loss : {avg_train_loss:.4f}"
        f"      Validation Loss : {avg_validation_loss:.4f}"
    )
    
    
    # 2(pt, momentum) physical parameters are pending 

 Validation loss decreased (inf to 0.2415) Saving best model
Epoch 1/80      Train Loss : 0.3573      Validation Loss : 0.2415
 Validation loss decreased (0.2415 to 0.2340) Saving best model
Epoch 2/80      Train Loss : 0.2720      Validation Loss : 0.2340
 Validation loss decreased (0.2340 to 0.2305) Saving best model
Epoch 3/80      Train Loss : 0.2665      Validation Loss : 0.2305
 Validation loss decreased (0.2305 to 0.2276) Saving best model
Epoch 4/80      Train Loss : 0.2605      Validation Loss : 0.2276
 Validation loss decreased (0.2276 to 0.2222) Saving best model
Epoch 5/80      Train Loss : 0.2611      Validation Loss : 0.2222
 Validation loss decreased (0.2222 to 0.2185) Saving best model
Epoch 6/80      Train Loss : 0.2591      Validation Loss : 0.2185
Epoch 7/80      Train Loss : 0.2521      Validation Loss : 0.2198
 Validation loss decreased (0.2185 to 0.2182) Saving best model
Epoch 8/80      Train Loss : 0.2510      Validation Loss : 0.2182
 Validation loss decreased 

In [45]:
# best model load

model = trackGAT()
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dictionary'])

C:\Users\js731\AppData\Local\Temp\ipykernel_24336\2347601437.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(best_model_path)


<All keys matched successfully>

In [46]:
model.eval()

test_loss = 0

with torch.no_grad():

    for batch in test_loader:

        prediction = model(
            batch.x,
            batch.edge_index,
            batch.batch
        )

        loss = criterion(
            prediction,
            batch.y
        )

        test_loss += loss.item()


print("Test Loss:", test_loss/len(test_loader))

Test Loss: 0.11945524057215554


In [ ]:
# results analysis ()



In [ ]:
# 21 july 2026
# 1 best model save 
# 2 fix the validation loss bug 
# 3 but the training loss and validation loss is little bit fluctuating
# 4 pt, momentum are pending (que:- how to predict pt and momentum with the help of formula or by the model itself)
# analysis of results are pending 